# Crypto Trader Performance vs Market Sentiment Analysis

**Objective**: Analyze how Bitcoin market sentiment (Fear/Greed) relates to trader behavior and performance on Hyperliquid.

**Author**: Data Science Intern  
**Date**: February 2026

---

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print('Libraries loaded successfully.')

---
# Part A — Data Preparation

This section covers loading, cleaning, and preparing both datasets for analysis.

## A.1 Load Datasets

In [ ]:
# Load Fear/Greed Index data
fear_greed_df = pd.read_csv('../data/fear_greed_index.csv')

print('=== Fear/Greed Index Dataset ===')
print(f'Shape: {fear_greed_df.shape[0]:,} rows × {fear_greed_df.shape[1]} columns')
print(f'\nColumns: {list(fear_greed_df.columns)}')
print(f'\nData Types:\n{fear_greed_df.dtypes}')
print(f'\nFirst 5 rows:')
fear_greed_df.head()

In [ ]:
# Load Hyperliquid trader data
trades_df = pd.read_csv('../data/historical_data.csv')

print('=== Hyperliquid Trader Dataset ===')
print(f'Shape: {trades_df.shape[0]:,} rows × {trades_df.shape[1]} columns')
print(f'\nColumns: {list(trades_df.columns)}')
print(f'\nData Types:\n{trades_df.dtypes}')
print(f'\nFirst 5 rows:')
trades_df.head()

## A.2 Data Quality Assessment

In [ ]:
def assess_data_quality(df, name):
    """Check missing values and duplicates for a dataframe."""
    print(f'=== {name} Data Quality ===')
    
    # Missing values
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_summary = pd.DataFrame({
        'Missing Count': missing,
        'Missing %': missing_pct
    })
    print('\nMissing Values:')
    print(missing_summary[missing_summary['Missing Count'] > 0] if missing.sum() > 0 else 'No missing values found.')
    
    # Duplicates
    duplicates = df.duplicated().sum()
    print(f'\nDuplicate Rows: {duplicates:,} ({duplicates/len(df)*100:.2f}%)')
    
    return missing, duplicates

fg_missing, fg_dups = assess_data_quality(fear_greed_df, 'Fear/Greed Index')
print('\n' + '='*50 + '\n')
trades_missing, trades_dups = assess_data_quality(trades_df, 'Hyperliquid Trades')

### Handling Missing Values & Duplicates

**For Fear/Greed Index:**
- Each day should have exactly one sentiment reading
- Duplicates removed by keeping the first occurrence

**For Hyperliquid Trades:**
- Missing values in non-critical columns (like Transaction Hash) are acceptable
- Rows with missing critical values (Account, Closed PnL, Side) are removed
- Exact duplicates are removed as they likely represent data loading issues

In [ ]:
# Clean Fear/Greed data
fear_greed_clean = fear_greed_df.drop_duplicates(subset=['date'], keep='first').copy()
print(f'Fear/Greed: Removed {len(fear_greed_df) - len(fear_greed_clean)} duplicate dates')

# Clean trades data
critical_cols = ['Account', 'Closed PnL', 'Side', 'Size USD']
trades_clean = trades_df.dropna(subset=[col for col in critical_cols if col in trades_df.columns]).copy()
trades_clean = trades_clean.drop_duplicates()
print(f'Trades: Removed {len(trades_df) - len(trades_clean)} rows (missing values + duplicates)')
print(f'\nCleaned dataset sizes:')
print(f'  Fear/Greed: {len(fear_greed_clean):,} rows')
print(f'  Trades: {len(trades_clean):,} rows')

## A.3 Timestamp Conversion & Daily Alignment

**Justification for Daily Aggregation:**
- The Fear/Greed Index is published once per day, making daily the natural alignment granularity
- Daily aggregation smooths out intraday noise and allows for meaningful behavioral patterns
- Aligning at daily level enables direct comparison: "How did traders behave on Fear days vs Greed days?"

In [ ]:
# Convert Fear/Greed dates
fear_greed_clean['date'] = pd.to_datetime(fear_greed_clean['date'])
fear_greed_clean = fear_greed_clean.sort_values('date').reset_index(drop=True)

print('Fear/Greed Index date range:')
print(f'  From: {fear_greed_clean["date"].min()}')
print(f'  To: {fear_greed_clean["date"].max()}')
print(f'  Unique dates: {fear_greed_clean["date"].nunique()}')

In [ ]:
# Convert trades timestamps
# Try parsing the Timestamp IST column first
if 'Timestamp IST' in trades_clean.columns:
    trades_clean['datetime'] = pd.to_datetime(trades_clean['Timestamp IST'], errors='coerce')
elif 'Timestamp' in trades_clean.columns:
    # If numeric timestamp, convert from milliseconds
    trades_clean['datetime'] = pd.to_datetime(trades_clean['Timestamp'], unit='ms', errors='coerce')

# Extract date for daily aggregation
trades_clean['date'] = trades_clean['datetime'].dt.date
trades_clean['date'] = pd.to_datetime(trades_clean['date'])

print('Trades date range:')
print(f'  From: {trades_clean["date"].min()}')
print(f'  To: {trades_clean["date"].max()}')
print(f'  Unique dates: {trades_clean["date"].nunique()}')
print(f'  Unique accounts: {trades_clean["Account"].nunique()}')

In [ ]:
# Find overlapping date range
min_date = max(fear_greed_clean['date'].min(), trades_clean['date'].min())
max_date = min(fear_greed_clean['date'].max(), trades_clean['date'].max())

print(f'Overlapping date range for analysis:')
print(f'  From: {min_date}')
print(f'  To: {max_date}')

# Filter both datasets to overlapping period
fear_greed_aligned = fear_greed_clean[
    (fear_greed_clean['date'] >= min_date) & 
    (fear_greed_clean['date'] <= max_date)
].copy()

trades_aligned = trades_clean[
    (trades_clean['date'] >= min_date) & 
    (trades_clean['date'] <= max_date)
].copy()

print(f'\nAligned datasets:')
print(f'  Fear/Greed days: {len(fear_greed_aligned)}')
print(f'  Trades in period: {len(trades_aligned):,}')

## A.4 Create Key Metrics

We compute the following metrics per trader per day:
1. **Daily PnL**: Sum of Closed PnL for all trades
2. **Win Rate**: Percentage of profitable trades (Closed PnL > 0)
3. **Average Trade Size**: Mean of Size USD
4. **Trade Frequency**: Number of trades
5. **Long/Short Ratio**: Number of Long trades / Number of Short trades

In [ ]:
# Ensure numeric types
trades_aligned['Closed PnL'] = pd.to_numeric(trades_aligned['Closed PnL'], errors='coerce')
trades_aligned['Size USD'] = pd.to_numeric(trades_aligned['Size USD'], errors='coerce')

# Create daily metrics per trader
def compute_trader_daily_metrics(df):
    """Compute daily metrics for each trader."""
    
    # Group by account and date
    daily_metrics = df.groupby(['Account', 'date']).agg(
        daily_pnl=('Closed PnL', 'sum'),
        total_trades=('Closed PnL', 'count'),
        winning_trades=('Closed PnL', lambda x: (x > 0).sum()),
        avg_trade_size=('Size USD', 'mean'),
        total_volume=('Size USD', 'sum'),
        long_trades=('Side', lambda x: (x.str.upper() == 'BUY').sum() + (x.str.upper() == 'LONG').sum()),
        short_trades=('Side', lambda x: (x.str.upper() == 'SELL').sum() + (x.str.upper() == 'SHORT').sum()),
    ).reset_index()
    
    # Calculate win rate and long/short ratio
    daily_metrics['win_rate'] = daily_metrics['winning_trades'] / daily_metrics['total_trades']
    daily_metrics['long_short_ratio'] = daily_metrics['long_trades'] / daily_metrics['short_trades'].replace(0, np.nan)
    
    return daily_metrics

trader_daily = compute_trader_daily_metrics(trades_aligned)
print(f'Daily metrics computed for {trader_daily["Account"].nunique()} unique traders')
print(f'Total trader-day observations: {len(trader_daily):,}')
print(f'\nSample metrics:')
trader_daily.head(10)

In [ ]:
# Merge with sentiment data
# Simplify classification to Fear vs Greed
fear_greed_aligned['sentiment'] = fear_greed_aligned['classification'].apply(
    lambda x: 'Fear' if 'fear' in x.lower() else 'Greed'
)

# Merge
analysis_df = trader_daily.merge(
    fear_greed_aligned[['date', 'value', 'classification', 'sentiment']],
    on='date',
    how='left'
)

print(f'Final analysis dataset: {len(analysis_df):,} rows')
print(f'\nSentiment distribution in trading data:')
print(analysis_df['sentiment'].value_counts())
print(f'\nSample merged data:')
analysis_df.head()

In [ ]:
# Summary statistics of key metrics
metrics_summary = analysis_df[['daily_pnl', 'win_rate', 'avg_trade_size', 'total_trades', 'long_short_ratio']].describe()
print('=== Key Metrics Summary ===')
metrics_summary.round(2)

---
# Part B — Analysis

We now investigate the relationship between market sentiment and trader performance/behavior.

## B.1 Performance Comparison: Fear vs Greed Days

**Key Question**: Does trader performance differ between Fear and Greed days?

In [ ]:
# Remove outliers for cleaner visualization (keep data for stats)
def remove_outliers(df, column, n_std=3):
    """Remove outliers beyond n standard deviations."""
    mean = df[column].mean()
    std = df[column].std()
    return df[(df[column] >= mean - n_std * std) & (df[column] <= mean + n_std * std)]

# Performance metrics by sentiment
performance_by_sentiment = analysis_df.groupby('sentiment').agg({
    'daily_pnl': ['mean', 'median', 'std', lambda x: x.quantile(0.25)],
    'win_rate': ['mean', 'median'],
    'total_trades': ['mean', 'sum']
}).round(4)

print('=== Performance by Sentiment ===')
performance_by_sentiment

In [ ]:
# Statistical test for PnL difference
fear_pnl = analysis_df[analysis_df['sentiment'] == 'Fear']['daily_pnl'].dropna()
greed_pnl = analysis_df[analysis_df['sentiment'] == 'Greed']['daily_pnl'].dropna()

# Mann-Whitney U test (non-parametric, robust to non-normal distributions)
stat, pvalue = stats.mannwhitneyu(fear_pnl, greed_pnl, alternative='two-sided')

print('=== Statistical Test: PnL by Sentiment ===')
print(f'Mann-Whitney U statistic: {stat:,.0f}')
print(f'P-value: {pvalue:.4e}')
print(f'\nInterpretation: {"Significant difference" if pvalue < 0.05 else "No significant difference"} at α=0.05')

In [ ]:
# Visualization: PnL Distribution by Sentiment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot (clipped for visibility)
analysis_viz = remove_outliers(analysis_df, 'daily_pnl', n_std=2)
sns.boxplot(data=analysis_viz, x='sentiment', y='daily_pnl', ax=axes[0], palette=['#e74c3c', '#2ecc71'])
axes[0].set_title('Daily PnL Distribution by Sentiment', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Market Sentiment')
axes[0].set_ylabel('Daily PnL (USD)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Violin plot for win rate
sns.violinplot(data=analysis_df, x='sentiment', y='win_rate', ax=axes[1], palette=['#e74c3c', '#2ecc71'])
axes[1].set_title('Win Rate Distribution by Sentiment', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Market Sentiment')
axes[1].set_ylabel('Win Rate')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% Win Rate')

plt.tight_layout()
plt.savefig('../outputs/charts/pnl_sentiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Downside Risk Proxy: Percentage of losing days and average loss on losing days
downside_risk = analysis_df.groupby('sentiment').apply(
    lambda x: pd.Series({
        'loss_day_pct': (x['daily_pnl'] < 0).mean() * 100,
        'avg_loss_on_losing_day': x[x['daily_pnl'] < 0]['daily_pnl'].mean(),
        'worst_5pct_pnl': x['daily_pnl'].quantile(0.05),
        'pnl_volatility': x['daily_pnl'].std()
    })
).round(2)

print('=== Downside Risk Proxy by Sentiment ===')
downside_risk

### Key Finding: Performance by Sentiment

The analysis above shows how trader performance metrics differ between Fear and Greed market conditions. Key observations will be summarized in the final insights section.

## B.2 Behavioral Changes by Sentiment

**Key Question**: Do traders change their behavior based on market sentiment?

In [ ]:
# Behavioral metrics by sentiment
behavior_by_sentiment = analysis_df.groupby('sentiment').agg({
    'total_trades': ['mean', 'median'],
    'avg_trade_size': ['mean', 'median'],
    'long_short_ratio': ['mean', 'median'],
    'total_volume': ['mean', 'median']
}).round(4)

print('=== Behavioral Metrics by Sentiment ===')
behavior_by_sentiment

In [ ]:
# Visualization: Behavioral changes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trade frequency
analysis_viz = remove_outliers(analysis_df, 'total_trades', n_std=2)
sns.boxplot(data=analysis_viz, x='sentiment', y='total_trades', ax=axes[0, 0], palette=['#e74c3c', '#2ecc71'])
axes[0, 0].set_title('Trade Frequency by Sentiment', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Market Sentiment')
axes[0, 0].set_ylabel('Trades per Day')

# Position size
analysis_viz = remove_outliers(analysis_df, 'avg_trade_size', n_std=2)
sns.boxplot(data=analysis_viz, x='sentiment', y='avg_trade_size', ax=axes[0, 1], palette=['#e74c3c', '#2ecc71'])
axes[0, 1].set_title('Average Trade Size by Sentiment', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Market Sentiment')
axes[0, 1].set_ylabel('Avg Trade Size (USD)')

# Long/Short ratio
analysis_viz = analysis_df[analysis_df['long_short_ratio'].notna() & (analysis_df['long_short_ratio'] < 10)]
sns.boxplot(data=analysis_viz, x='sentiment', y='long_short_ratio', ax=axes[1, 0], palette=['#e74c3c', '#2ecc71'])
axes[1, 0].set_title('Long/Short Ratio by Sentiment', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Market Sentiment')
axes[1, 0].set_ylabel('Long/Short Ratio')
axes[1, 0].axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='Neutral (1:1)')

# Daily volume
analysis_viz = remove_outliers(analysis_df, 'total_volume', n_std=2)
sns.boxplot(data=analysis_viz, x='sentiment', y='total_volume', ax=axes[1, 1], palette=['#e74c3c', '#2ecc71'])
axes[1, 1].set_title('Daily Trading Volume by Sentiment', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Market Sentiment')
axes[1, 1].set_ylabel('Total Volume (USD)')

plt.tight_layout()
plt.savefig('../outputs/charts/behavior_sentiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical tests for behavioral differences
behavioral_tests = {}

for metric in ['total_trades', 'avg_trade_size', 'long_short_ratio']:
    fear_vals = analysis_df[analysis_df['sentiment'] == 'Fear'][metric].dropna()
    greed_vals = analysis_df[analysis_df['sentiment'] == 'Greed'][metric].dropna()
    
    if len(fear_vals) > 0 and len(greed_vals) > 0:
        stat, pvalue = stats.mannwhitneyu(fear_vals, greed_vals, alternative='two-sided')
        behavioral_tests[metric] = {
            'U-statistic': stat,
            'p-value': pvalue,
            'significant': pvalue < 0.05,
            'fear_median': fear_vals.median(),
            'greed_median': greed_vals.median()
        }

behavioral_tests_df = pd.DataFrame(behavioral_tests).T
print('=== Statistical Tests for Behavioral Differences ===')
behavioral_tests_df

## B.3 Trader Segmentation

We segment traders into intuitive groups to understand how sentiment affects different trader types.

In [ ]:
# Compute overall trader profile (across all days)
trader_profiles = analysis_df.groupby('Account').agg({
    'daily_pnl': ['sum', 'mean', 'std', 'count'],
    'win_rate': 'mean',
    'total_trades': ['sum', 'mean'],
    'avg_trade_size': 'mean',
    'long_short_ratio': 'mean'
}).reset_index()

# Flatten column names
trader_profiles.columns = ['Account', 'total_pnl', 'avg_daily_pnl', 'pnl_volatility', 'active_days',
                           'avg_win_rate', 'total_trades', 'avg_daily_trades', 'avg_trade_size', 'avg_ls_ratio']

print(f'Trader profiles computed for {len(trader_profiles)} unique traders')
trader_profiles.head()

In [ ]:
# Segment 1: Trading Frequency (Frequent vs Infrequent)
freq_median = trader_profiles['avg_daily_trades'].median()
trader_profiles['frequency_segment'] = trader_profiles['avg_daily_trades'].apply(
    lambda x: 'Frequent' if x >= freq_median else 'Infrequent'
)

# Segment 2: Position Size (Large vs Small)
size_median = trader_profiles['avg_trade_size'].median()
trader_profiles['size_segment'] = trader_profiles['avg_trade_size'].apply(
    lambda x: 'Large Positions' if x >= size_median else 'Small Positions'
)

# Segment 3: Consistency (Consistent vs Inconsistent based on PnL volatility relative to returns)
# Using coefficient of variation (CV) as consistency measure
trader_profiles['consistency_cv'] = trader_profiles['pnl_volatility'] / (abs(trader_profiles['avg_daily_pnl']) + 1)
cv_median = trader_profiles['consistency_cv'].median()
trader_profiles['consistency_segment'] = trader_profiles['consistency_cv'].apply(
    lambda x: 'Consistent' if x <= cv_median else 'Inconsistent'
)

print('=== Trader Segmentation Summary ===')
print(f'\nFrequency Segment (split at {freq_median:.1f} trades/day):')
print(trader_profiles['frequency_segment'].value_counts())
print(f'\nSize Segment (split at ${size_median:,.0f}):')
print(trader_profiles['size_segment'].value_counts())
print(f'\nConsistency Segment (split at CV={cv_median:.2f}):')
print(trader_profiles['consistency_segment'].value_counts())

In [ ]:
# Merge segments back to analysis dataframe
analysis_segmented = analysis_df.merge(
    trader_profiles[['Account', 'frequency_segment', 'size_segment', 'consistency_segment']],
    on='Account',
    how='left'
)

print(f'Segmented analysis dataset: {len(analysis_segmented):,} rows')

In [ ]:
# Analyze performance by segment and sentiment
def segment_sentiment_analysis(df, segment_col, segment_name):
    """Analyze how sentiment impacts different trader segments."""
    result = df.groupby([segment_col, 'sentiment']).agg({
        'daily_pnl': ['mean', 'median'],
        'win_rate': 'mean',
        'total_trades': 'mean'
    }).round(4)
    print(f'\n=== {segment_name} by Sentiment ===')
    return result

freq_analysis = segment_sentiment_analysis(analysis_segmented, 'frequency_segment', 'Trading Frequency')
print(freq_analysis)

size_analysis = segment_sentiment_analysis(analysis_segmented, 'size_segment', 'Position Size')
print(size_analysis)

consistency_analysis = segment_sentiment_analysis(analysis_segmented, 'consistency_segment', 'Consistency')
print(consistency_analysis)

In [ ]:
# Visualization: Segment performance by sentiment
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Frequency segment
analysis_viz = remove_outliers(analysis_segmented, 'daily_pnl', n_std=2)
sns.barplot(data=analysis_viz, x='frequency_segment', y='daily_pnl', hue='sentiment', 
            ax=axes[0], palette=['#e74c3c', '#2ecc71'], errorbar='ci')
axes[0].set_title('PnL by Trading Frequency & Sentiment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Trader Type')
axes[0].set_ylabel('Mean Daily PnL (USD)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].legend(title='Sentiment')

# Size segment
sns.barplot(data=analysis_viz, x='size_segment', y='daily_pnl', hue='sentiment', 
            ax=axes[1], palette=['#e74c3c', '#2ecc71'], errorbar='ci')
axes[1].set_title('PnL by Position Size & Sentiment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trader Type')
axes[1].set_ylabel('Mean Daily PnL (USD)')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].legend(title='Sentiment')

# Consistency segment
sns.barplot(data=analysis_viz, x='consistency_segment', y='daily_pnl', hue='sentiment', 
            ax=axes[2], palette=['#e74c3c', '#2ecc71'], errorbar='ci')
axes[2].set_title('PnL by Consistency & Sentiment', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Trader Type')
axes[2].set_ylabel('Mean Daily PnL (USD)')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].legend(title='Sentiment')

plt.tight_layout()
plt.savefig('../outputs/charts/segment_sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## B.4 Key Insights (Evidence-Backed)

We now synthesize our findings into actionable insights.

In [ ]:
# Insight 1: Long/Short bias shifts with sentiment
ls_by_sentiment = analysis_df.groupby('sentiment')['long_short_ratio'].agg(['mean', 'median', 'std'])
print('=== Insight 1: Long/Short Ratio by Sentiment ===')
print(ls_by_sentiment)
print('\nInterpretation: Examine whether traders lean more long during Greed and more short during Fear.')

In [ ]:
# Insight 2: Win rate consistency across sentiment
print('=== Insight 2: Win Rate Stability ===')
wr_stats = analysis_df.groupby('sentiment')['win_rate'].agg(['mean', 'std']).round(4)
print(wr_stats)

# Coefficient of variation for win rate
wr_stats['cv'] = wr_stats['std'] / wr_stats['mean']
print('\nWin Rate Coefficient of Variation:')
print(wr_stats['cv'])

In [ ]:
# Insight 3: Relationship between sentiment value (fear/greed intensity) and performance
# Correlate the numeric sentiment value with daily PnL
corr_pnl = analysis_df['value'].corr(analysis_df['daily_pnl'])
corr_wr = analysis_df['value'].corr(analysis_df['win_rate'])
corr_freq = analysis_df['value'].corr(analysis_df['total_trades'])

print('=== Insight 3: Correlation with Sentiment Index Value ===')
print(f'Correlation with Daily PnL: {corr_pnl:.4f}')
print(f'Correlation with Win Rate: {corr_wr:.4f}')
print(f'Correlation with Trade Frequency: {corr_freq:.4f}')

In [ ]:
# Insight 4: Extreme sentiment days
# Define extreme fear (<25) and extreme greed (>75)
analysis_df['sentiment_extreme'] = analysis_df['value'].apply(
    lambda x: 'Extreme Fear' if x <= 25 else ('Extreme Greed' if x >= 75 else 'Moderate')
)

extreme_analysis = analysis_df.groupby('sentiment_extreme').agg({
    'daily_pnl': ['mean', 'median', 'std'],
    'win_rate': ['mean'],
    'total_trades': ['mean']
}).round(4)

print('=== Insight 4: Performance on Extreme Sentiment Days ===')
print(extreme_analysis)

In [ ]:
# Visualization: Extreme sentiment impact
fig, ax = plt.subplots(figsize=(10, 6))

order = ['Extreme Fear', 'Moderate', 'Extreme Greed']
available_categories = [cat for cat in order if cat in analysis_df['sentiment_extreme'].unique()]

analysis_viz = remove_outliers(analysis_df, 'daily_pnl', n_std=2)
sns.boxplot(data=analysis_viz, x='sentiment_extreme', y='daily_pnl', 
            order=available_categories, palette=['#c0392b', '#3498db', '#27ae60'], ax=ax)
ax.set_title('Daily PnL by Sentiment Intensity', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Category')
ax.set_ylabel('Daily PnL (USD)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/charts/extreme_sentiment_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part C — Actionable Output

Based on our analysis, we propose practical strategy rules and risk controls.

In [ ]:
# Generate summary statistics for strategy recommendations
print('=== Data Summary for Strategy Recommendations ===')

# Fear days stats
fear_data = analysis_df[analysis_df['sentiment'] == 'Fear']
greed_data = analysis_df[analysis_df['sentiment'] == 'Greed']

print(f'\nFear Days ({len(fear_data):,} observations):')
print(f'  Mean PnL: ${fear_data["daily_pnl"].mean():,.2f}')
print(f'  Median PnL: ${fear_data["daily_pnl"].median():,.2f}')
print(f'  Win Rate: {fear_data["win_rate"].mean():.2%}')
print(f'  Avg Trade Size: ${fear_data["avg_trade_size"].mean():,.0f}')

print(f'\nGreed Days ({len(greed_data):,} observations):')
print(f'  Mean PnL: ${greed_data["daily_pnl"].mean():,.2f}')
print(f'  Median PnL: ${greed_data["daily_pnl"].median():,.2f}')
print(f'  Win Rate: {greed_data["win_rate"].mean():.2%}')
print(f'  Avg Trade Size: ${greed_data["avg_trade_size"].mean():,.0f}')

In [ ]:
# Segment-specific insights for recommendations
print('=== Segment-Specific Performance Summary ===')

for segment in ['frequency_segment', 'size_segment', 'consistency_segment']:
    print(f'\n{segment.replace("_", " ").title()}:')
    seg_summary = analysis_segmented.groupby([segment, 'sentiment'])['daily_pnl'].agg(['mean', 'count']).round(2)
    print(seg_summary)

## Strategy Recommendations

Based on the analysis above, we present two practical strategy rules:

### Strategy Rule 1: Sentiment-Based Position Sizing

**Observation**: [To be filled based on actual data patterns]

**Recommendation**: Adjust position sizes based on the Fear/Greed index value.

### Strategy Rule 2: Segment-Specific Risk Controls

**Observation**: [To be filled based on actual data patterns]

**Recommendation**: Apply differentiated risk limits based on trader profiles.

---

*Detailed recommendations are documented in the summary.md file.*

---
# Summary & Key Takeaways

This analysis explored the relationship between Bitcoin market sentiment and trader performance on Hyperliquid.

**Methodology:**
1. Aligned two datasets (Fear/Greed Index + Hyperliquid trades) at daily granularity
2. Created key performance metrics (PnL, win rate, trade size, frequency, long/short ratio)
3. Compared metrics between Fear and Greed market conditions
4. Segmented traders by frequency, position size, and consistency
5. Analyzed how different segments respond to sentiment

**Key Findings:**
- [Summary of findings to be documented after running the notebook]

**Limitations:**
- Daily aggregation may mask intraday patterns
- No leverage data available in this dataset
- Sentiment index is Bitcoin-focused but traders may trade other assets

**Next Steps:**
- Implement proposed strategy rules in a backtesting framework
- Monitor real-time sentiment for position management signals

In [ ]:
# Save processed data for future reference
analysis_segmented.to_csv('../outputs/processed_analysis_data.csv', index=False)
trader_profiles.to_csv('../outputs/trader_profiles.csv', index=False)
print('Processed data saved to outputs folder.')
print('\nAnalysis complete.')